# Import and load data

In [26]:
import pandas as pd
dataset = pd.read_csv('IndianFinancialNews.csv', index_col='Unnamed: 0')
dataset


,Date,Title,Description
0,"May 26, 2020, Tuesday","ATMs to become virtual bank branches, accept d...","Close to 14.6 per cent (or 35,000) of the 240,..."
1,"May 26, 2020, Tuesday",IDFC First Bank seniors to forgo 65% of bonus ...,"V Vaidyanathan, managing director and chief ex..."
2,"May 25, 2020, Monday","Huge scam in YES Bank for many years, says Enf...",Rana Kapoor's wife also charged with abetting ...
3,"May 24, 2020, Sunday","Bank of Maharashtra sanctioned Rs 2,789 cr in ...",The bank said it was now gearing up to extend ...
4,"May 23, 2020, Saturday",DCB Bank's profit before tax declines 37.6% to...,Net profit for the financial year ended March ...
...,...,...,...
49995,"February 11, 2003, Tuesday",Lic Mops Up Government Securities As Prices Crash,Lic Mops Up Government Securities As Prices Crash
49996,"February 11, 2003, Tuesday",Banks Plan To Raise Lending Rates Without Alte...,Banks Plan To Raise Lending Rates Without Alte...
49997,"February 10, 2003, Monday","Net Scheduled Inflows Of Rs 1,559.9 Crore","Net Scheduled Inflows Of Rs 1,559.9 Crore"
49998,"February 10, 2003, Monday",Rbi Calls Meet To Push Floating Rate Deposits,Rbi Calls Meet To Push Floating Rate Deposits


In [27]:
dataset.info()


<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         50000 non-null  object
 1   Title        50000 non-null  object
 2   Description  49290 non-null  object
dtypes: object(3)
memory usage: 1.5+ MB


# Sentiment Analyzer

In [30]:

def senti_analyze(dataset, column_name):

    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    
    dataset['scores'] = dataset[column_name].apply(lambda headline: sia.polarity_scores(headline))
    dataset['compound'] = dataset['scores'].apply(lambda score_dict: score_dict['compound'])
    dataset['negative'] = dataset['scores'].apply(lambda score_dict: score_dict['neg'])
    dataset['neutral'] = dataset['scores'].apply(lambda score_dict: score_dict['neu'])
    dataset['positive'] = dataset['scores'].apply(lambda score_dict: score_dict['pos'])
    dataset['comp_score'] = dataset['compound'].apply(lambda compscore: 'positive' if compscore >=0 else 'negative')

    return dataset

senti_analyze(dataset, 'Description')


,Date,Title,Description,scores,compound,negative,neutral,positive,comp_score
0,"May 26, 2020, Tuesday","ATMs to become virtual bank branches, accept d...","Close to 14.6 per cent (or 35,000) of the 240,...","{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000,positive
1,"May 26, 2020, Tuesday",IDFC First Bank seniors to forgo 65% of bonus ...,"V Vaidyanathan, managing director and chief ex...","{'neg': 0.11, 'neu': 0.89, 'pos': 0.0, 'compou...",-0.2732,0.110,0.890,0.000,negative
2,"May 25, 2020, Monday","Huge scam in YES Bank for many years, says Enf...",Rana Kapoor's wife also charged with abetting ...,"{'neg': 0.469, 'neu': 0.531, 'pos': 0.0, 'comp...",-0.6486,0.469,0.531,0.000,negative
3,"May 24, 2020, Sunday","Bank of Maharashtra sanctioned Rs 2,789 cr in ...",The bank said it was now gearing up to extend ...,"{'neg': 0.0, 'neu': 0.925, 'pos': 0.075, 'comp...",0.1779,0.000,0.925,0.075,positive
4,"May 23, 2020, Saturday",DCB Bank's profit before tax declines 37.6% to...,Net profit for the financial year ended March ...,"{'neg': 0.0, 'neu': 0.888, 'pos': 0.112, 'comp...",0.4404,0.000,0.888,0.112,positive
...,...,...,...,...,...,...,...,...,...
49995,"February 11, 2003, Tuesday",Lic Mops Up Government Securities As Prices Crash,Lic Mops Up Government Securities As Prices Crash,"{'neg': 0.248, 'neu': 0.55, 'pos': 0.202, 'com...",-0.1280,0.248,0.550,0.202,negative
49996,"February 11, 2003, Tuesday",Banks Plan To Raise Lending Rates Without Alte...,Banks Plan To Raise Lending Rates Without Alte...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000,positive
49997,"February 10, 2003, Monday","Net Scheduled Inflows Of Rs 1,559.9 Crore","Net Scheduled Inflows Of Rs 1,559.9 Crore","{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000,positive
49998,"February 10, 2003, Monday",Rbi Calls Meet To Push Floating Rate Deposits,Rbi Calls Meet To Push Floating Rate Deposits,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000,positive


In [33]:
dataset['comp_score'].value_counts()


comp_score
positive    40149
negative     9141
Name: count, dtype: int64

# Topic Modelling

In [56]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

for vectorizer in [CountVectorizer, TfidfVectorizer]:
    print('\n', vectorizer.__name__, '\n', '.'*20)
    tfidf=vectorizer(max_df=0.95,min_df=2,stop_words='english')
    dtm=tfidf.fit_transform(dataset['Description'])

    for models in [NMF, LatentDirichletAllocation]:
        print('\n', models.__name__)

        model=models(n_components=5,random_state=10)
        model.fit(dtm)

        for index,topic in enumerate(model.components_):
            results=([tfidf.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
            print(results)

        topic_results=model.transform(dtm)
        dataset[f"Topic_{models.__name__}_{vectorizer.__name__}"]=topic_results.argmax(axis=1)
        print(dataset[f"Topic_{models.__name__}_{vectorizer.__name__}"].value_counts())



 CountVectorizer 
 ....................

 NMF
['sbi', 'country', 'icici', 'sector', 'largest', 'lender', 'said', 'today', 'state', 'bank']
['reported', 'year', 'ended', '000', 'cent', 'quarter', 'profit', 'net', 'crore', 'rs']
['state', 'foreign', 'policy', 'governor', 'today', 'said', 'bank', 'rbi', 'reserve', 'india']
['authority', 'irda', 'lic', 'regulatory', 'development', 'companies', 'corporation', 'company', 'life', 'insurance']
['said', 'loans', 'credit', 'government', 'today', 'private', 'finance', 'public', 'sector', 'banks']
Topic_NMF_CountVectorizer
4    16677
0    11863
1     7899
3     6432
2     6419
Name: count, dtype: int64

 LatentDirichletAllocation
['today', 'rs', 'rates', 'quarter', 'profit', 'rate', 'india', 'net', 'cent', 'bank']
['today', 'largest', 'country', 'private', 'sector', 'state', 'life', 'india', 'insurance', 'bank']
['sector', 'banking', 'government', 'said', 'finance', 'india', 'reserve', 'rbi', 'bank', 'banks']
['indian', 'insurance', 'year', 'week